## About
- This notebook includes the data pre-processing steps of the ALD Olink liver inflammation biomarker study. Specifically, it includes
- 1. Olink plasma proteomics filtering
  2. clinical data wrangling

### Input: 
- 1. combined proteomics and clinical data
  2. MCV data 

### Output
- 1. Curated and combined proteomics and clinical data

In [ ]:
# Load packages
import pandas as pd
import numpy as np
import os
import pingouin as pg

## Clinical data

### Load clinical data (came in a combined format with proteomics data)

In [ ]:
Base = 'data_directory'
df_olink = pd.read_csv(os.path.join(Base, 'biomarker.csv')).drop(['Unnamed: 0'], axis=1).set_index('CBMRID')
df_mcv = pd.read_csv(os.path.join(Base, 'MCV_biomarker.txt'), sep='\t')
df_olink = df_olink.join(df_mcv.set_index('CBMRID'), how='left')

In [ ]:
# Check for duplicated index
duplicated_ids = df_olink.loc[df_olink.index.duplicated()].index
print("Number of duplicated ids is {}.".format(len(duplicated_ids)))

# if there are duplicated IDs
# one option is to rename duplicated index with the following:
# df_olink.index = df_olink.index.where(~df_olink.index.duplicated(), df_olink.index + '_dp')

In [ ]:
print('Input data contains proteomics and clinical data for {} participants, \namong which {} has inflam_binary score assigned'.format(len(df_olink), df_olink['inflam_binary'].count()))

In [ ]:
# Compute binary outcomes for inflammation levels ≥3 and 4
df_olink['inflam_binary_I3'] = df_olink['inflam_binary'].where(df_olink['inflammation'] != 2, 0)
df_olink['inflam_binary_I4'] = df_olink['inflam_binary_I3'].where(df_olink['inflammation'] != 3, 0)

In [ ]:
df_olink['inflam_binary'].value_counts()

In [ ]:
df_olink['Sample_type'].value_counts()

In [ ]:
df_olink.groupby('Sample_type')['cohort'].value_counts()

In [ ]:
# encode the cohort of participants with 'Sample_type' == 'plasma_LiHep' as 'GALA_LiHep'
df_olink.loc[df_olink['Sample_type']=='plasma_LiHep', 'cohort'] = 'GALA_LiHep'
# encode male to 1, and female to 0
df_olink['gender'] = df_olink['gender'].map({'male':1, 'female':0})
# encode availability of biopsy based on presence of histology scores
mask = df_olink[['steatosis', 'inflammation', 'kleiner']].isna().all(axis=1)
df_olink['biopsy']=np.where(mask, False, True)

### Encode participatns classified as "steatohepatitis at risk of progression" (NASH + NAS>=4, F>=2)

In [ ]:
# First annotate patients classified as NASH according to the FLIP algorithm
cols = ['steatosis', 'ballooning', 'lobinflammation']
for index, row in df_olink.iterrows():
    if not row[cols].isnull().values.any():
        cond1=row['steatosis']>0
        cond2=row['ballooning']>0
        cond3=row['lobinflammation']>0
        if cond1&cond2&cond3:
            df_olink.loc[index, 'nash']='NASH'
        elif not cond1:
            df_olink.loc[index, 'nash']='notNASH'
        else:
            df_olink.loc[index, 'nash']='Steatosis'
    else:
        df_olink.loc[index, 'nash'] = np.nan

In [ ]:
# Second, recode nash diagnosis for individuals with pure ALD risk (no presence of metabolic syndrome)
# for individuals with pure ALD risk, grade one steatosis is not required
df_olink['nash_new']=df_olink['nash']
cond1 = df_olink['aldrisk']==1
#cond2 = df_olink['mets']!=1
cond3 = df_olink['ballooning']>0
cond4 = df_olink['lobinflammation']>0
df_olink.loc[cond1&cond3&cond4, 'nash_new']='NASH'

# Print out consequence of re-classification based on pure ALD risk:
n_reclass = df_olink['nash_new'].value_counts()['NASH'] - df_olink['nash'].value_counts()['NASH']
print('{} individuals had been re-classified as NASH\nbased on pure ALD risk (presence of metabolic syndrome not required'.format(n_reclass))

In [ ]:
# Third, assign "steatohepatitis at risk of progression (NASH + NAS>=4 + F>=2)
cols = ['nash_new', 'nas', 'fibrosis']
for index, row in df_olink.iterrows():
    if not row[cols].isnull().values.any():
        cond1=row['nash_new']=='NASH'
        cond2=row['nas']>=4
        cond3=row['fibrosis'] in ['F2', 'F3', 'F4']
        if cond1&cond2&cond3:
            df_olink.loc[index, 'at_risk_progression']=1
        else:
            df_olink.loc[index, 'at_risk_progression']=0
    else:
        df_olink.loc[index, 'at_risk_progression'] = np.nan

### Compute FAST score, the state-of-the-art non-invasive measure of steatohepatitis at risk of progression

In [ ]:
# Compute FAST score according to https://doi.org/10.1016%2FS2468-1253(19)30383-8 
cols = ['te', 'cap', 'ast']
import math
for index, row in df_olink.iterrows():
    if not row[cols].isnull().values.any():
        lsm=row['te']
        cap=row['cap']
        enzyme_ast=row['ast']
        score=math.e**(-1.65 + 1.07*np.log(lsm) + 2.66*10**-8*cap**3 - 63.3*enzyme_ast**-1)
        fast_score=score/(1+score)
        df_olink.loc[index,'fast']=fast_score
    else:
        df_olink.loc[index, 'fast']=np.nan

In [ ]:
df_olink[['fast', 'at_risk_progression']].count()

### Compute ALD/NAFLD Index (ANI), defined as:
- -58.5 + 0.637 (MCV) + 3.91 (AST/ALT) – 0.406 (BMI) + 6.35 for male gender
-  with the correction:
-  mcv <-(92, 103), ast/alt <=3

In [ ]:
cols = ['mcv', 'bmi', 'ast_alt', 'gender']
min_mcv = 92
max_mcv = 103
max_aar = 3
for index, row in df_olink.iterrows():
    if not row[cols].isnull().values.any():
        mcv=row['mcv']
        new_mcv=min_mcv if mcv < min_mcv else max_mcv if mcv > max_mcv else mcv
        aar=row['ast_alt']
        new_aar=max_aar if aar > max_aar else aar
        gender=row['gender']
        bmi=row['bmi']
        ani_base=-58.5 + 0.637*new_mcv + 3.91*new_aar - 0.406*bmi 
        ani=ani_base if gender==0 else ani_base+6.35
        df_olink.loc[index, 'ani']=ani
    else:
        df_olink.loc[index, 'ani']=np.nan

df_olink['ani_abs']=abs(df_olink['ani'])

### Make additional labels: patients who have F2-4 and I1-5; patients who have ballooning only

In [ ]:
import numpy as np

df_olink['F2-4+I1-5'] = np.where(
    (df_olink['kleiner'] > 1) & (df_olink['inflammation'] > 0), 1,
    np.where((df_olink['kleiner'] > 1) & (df_olink['inflammation'] == 0), 0, np.nan)
)

df_olink['ballooning only'] = np.where(
    (df_olink['lobinflammation'] == 0) & (df_olink['ballooning'] > 0), 1,
    np.where((df_olink['lobinflammation'] > 0) & (df_olink['ballooning'] == 0), 0, np.nan)
)

### Assign 'normal' and 'supernormal' controls in the absence of biopsy based on sets of criteria (see src functio)

In [ ]:
from src.preprocess import assign_normal, assign_supernormal

In [ ]:
df_olink = assign_supernormal(assign_normal(df_olink))

In [ ]:
# Create a column of samples without a biopsy and are not supernormal
df_olink['supernormal_filter'] = (~df_olink['biopsy']) & (~df_olink['supernormal'])

### Log2 transform ALT and AST levels

In [ ]:
for i in ['alt', 'ast']:
    df_olink[i+'_log2']=df_olink[i].apply(np.log2)
addon = ['alt_log2', 'ast_log2']

### Dichotomize based on BMI and HbA1c

In [ ]:
df_olink['bmi>=30']=np.where(df_olink['bmi']<30,0, np.where(df_olink['bmi'].isna(), np.nan, 1))
df_olink['hba1c>=48mmol/mol']=np.where(df_olink['hba1c']<48,0, np.where(df_olink['hba1c'].isna(), np.nan, 1))

### Histological score distribution

In [ ]:
def count_distribution(data, para, group='cohort'):
    df=data.copy()
    df_count = pd.DataFrame(df.groupby(group)[para].value_counts()).rename({para:'count'}, axis=1)
    df_count = df_count.reset_index().pivot(index=group, columns=para)
    df_count['count', 'sum']=df_count.sum(axis=1)
    df_count.columns = [para + '_' + str(i) for i in [j[1] for j in df_count.columns]]
    return(df_count)

count_distribution(df_olink, 'inflam_binary')

### Load proteomics variable names and clinical parameters

In [ ]:
cytokines = pd.read_csv(os.path.join(Base, 'raw/cytokine_list.txt'), sep='\t')['Cytokines']
clinical_parameters = pd.read_csv(os.path.join(Base, 'raw/clinical_parameters.txt'), sep='\t')['clinical parameters']
grtarget_labels = pd.read_csv(os.path.join(Base, 'raw/labels.txt'), sep='\t')['histology scores']

## Proteomics data

In [ ]:
rdc_ids=df_olink[df_olink['cohort']=='RDC'].index
ald_ids=df_olink[df_olink['cohort']=='ALD'].index
hp_ids=df_olink[df_olink['cohort']=='HP'].index
sip_ids=df_olink[df_olink['cohort']=='SIP'].index
lihep_ids = df_olink[df_olink['cohort']=='GALA_LiHep'].index
sip_biop_ids = df_olink.iloc[df_olink.index.isin(sip_ids)][['lobinflammation', 'ballooning']].dropna().index

In [ ]:
df_cytokines=df_olink[cytokines]
df_cytokines.isnull().sum().sum()

In [ ]:
df_cytokines = df_cytokines.assign(new_col = df_cytokines.isnull().sum(axis=1))
df_cytokines = df_cytokines.rename({'new_col':'missing value'}, axis=1)

In [ ]:
# Count missing values per individual
df_cytokines.sort_values(by='missing value', ascending =False).head(5)

In [ ]:
# Remove individuals who have >40% missing vallues
THRESHOLD_PER_INDI = 0.6

In [ ]:
indices_to_drop = df_cytokines[df_cytokines['missing value']>df_cytokines.shape[1] * THRESHOLD_PER_INDI].index

In [ ]:
indices_to_drop

In [ ]:
df_cytokines.loc[indices_to_drop]

In [ ]:
df_cytokines = df_cytokines.drop(indices_to_drop, axis=0)

In [ ]:
print('{} samples had missing value > {}'.format(len(indices_to_drop), THRESHOLD_PER_INDI))

In [ ]:
df_cytokines.isnull().sum().sum()

In [ ]:
df_cytokines.loc[:, df_cytokines.isna().any()].head()

In [ ]:
cytokines_with_missingvalues = df_cytokines.loc[:, df_cytokines.isna().any()].columns
print("{} cytokines have missing values after filtering for {} completeness".format(len(cytokines_with_missingvalues), THRESHOLD_PER_INDI))

In [ ]:
corr = pg.pairwise_corr(data=df_olink, columns=[['kleiner', 'inflammation', 'steatosis'], list(cytokines)]).set_index('X')
corr['abs(r)']=abs(corr['r'])
corr=corr.sort_values(by='abs(r)', ascending = False)
corr['abs(r)_rank'] = np.arange(corr.shape[0])

- 11 cytokines have missing values adding to a total of 79
- check if imputation is needed for these cytokines
- check if these cytokines are correlated with any of the histological scores
- the largest correlation has a Pearson coefficient r of 0.30, lower than >30 cytokines in inflammation, decide to exclude.

In [ ]:
corr[corr['Y'].isin(cytokines_with_missingvalues)].sort_values(by='abs(r)', ascending=False).head(8)

In [ ]:
df_cytokines_filtered=df_cytokines.dropna(axis=1).drop(['missing value'], axis=1)
df_cytokines_filtered.shape

In [ ]:
cytokines_filtered=df_cytokines_filtered.columns
key_ProteinID_olink=pd.DataFrame(list(zip(cytokines_filtered, cytokines_filtered)), columns=['Protein ID', 'Gene names']).set_index('Protein ID')
print('{} cytokines remained after filtering for missing values.'.format(len(cytokines_filtered)))

#### Overlapping indexes

In [ ]:
all_cli = [i for i in df_olink.columns if i not in cytokines.tolist()]
df_cli = df_olink[all_cli]
df_cli = df_cli[df_cli.index.isin(df_cytokines_filtered.index)]

In [ ]:
df_olink.shape

In [ ]:
df_cytokines_filtered.shape

In [ ]:
df_cli.shape

## Save curated data

In [ ]:
# Save curated clinical and proteomics data
with pd.ExcelWriter(os.path.join(Base, 'data_curated/cureated_proteomics_clinical_data.xlsx')) as writer:
    df_cli.to_excel(writer, sheet_name='clinical_curated')
    df_cytokines_filtered.to_excel(writer, sheet_name='proteomics_curated') 